In [1]:
import pandas as pd
import numpy as np

#Part 1 — Clean and Prepare the Data

In [2]:
# Load the dataset
hs = pd.read_csv('historical_stocks.csv')
hsp = pd.read_csv('historical_stock_prices.csv')

In [3]:
#check the shpe of dat set
print('historical_stocks')
print(hs.shape)
print('historical_stock_prices')
print(hsp.shape)

historical_stocks
(6460, 5)
historical_stock_prices
(20973889, 8)


In [4]:
#change the date datatpe type
hsp['date'] = pd.to_datetime(hsp['date'])
print(hsp.dtypes['date'])

datetime64[us]


In [5]:
#i have checked that does the that junk column is in the HSP dataset
print("'Symbol' rows in hsp:", (hsp['ticker'] == 'Symbol').sum())

'Symbol' rows in hsp: 0


In [6]:
# drop the 'Symbol' junk record
hs = hs[hs['ticker'] != 'Symbol'].reset_index(drop=True)
print("shape now:", hs.shape)

shape now: (6459, 5)


In [7]:
#short the stock price by ticker and date
hsp = hsp.sort_values(by=['ticker', 'date'])
print(hsp.head())
print(hsp.tail())

         ticker       open      close  adj_close        low       high  \
13766141      A  32.546494  31.473534  27.494957  28.612303  35.765381   
13766149      A  30.713520  28.880543  25.229753  28.478184  30.758226   
13766150      A  29.551144  31.473534  27.494957  28.657009  31.473534   
13766151      A  30.400572  28.612303  24.995413  28.612303  31.205294   
13766152      A  28.701717  29.372318  25.659359  28.612303  29.998211   

            volume       date  
13766141  62546300 1999-11-18  
13766149  15234100 1999-11-19  
13766150   6577800 1999-11-22  
13766151   5975600 1999-11-23  
13766152   4843200 1999-11-24  
         ticker  open  close  adj_close    low   high  volume       date
10905136   ZYNE  6.28   6.13       6.13  6.010  6.339  309100 2018-08-20
10905148   ZYNE  6.13   6.31       6.31  6.070  6.500  589300 2018-08-21
10905156   ZYNE  6.31   6.26       6.26  6.235  6.500  263200 2018-08-22
10905169   ZYNE  6.40   6.30       6.30  6.160  6.460  325900 2018-08-2

###Interpretation
I loaded both datasets and confirmed the prices file now has about 21 million rows (the full dataset). I converted the date column to datetime so I can work with it properly. I then applied the cleaning decision from Project 1 and removed the accidental "Symbol" junk record from the stocks file, which brought it from 6,460 to 6,459 rows. I also checked that this junk ticker did not exist in the prices file, and it returned 0, so the prices data was already clean of it. Finally, I sorted the price data by ticker and then by date, so that each stock's rows are in time order — this is essential before creating features like previous-day price and moving averages.

###Decision on unusual close and adj_close values:
 I reviewed the unusual high values I found in Project 1 (the KTOS spike of about $1,578 in March 2000). I decided to retain and flag these records. They are genuine, real stock prices, not errors — the spike appears in both close and adj_close for the same stock and moves smoothly across consecutive days, which is how real price movements behave. Deleting real data just to make the statistics look smaller would bias the dataset, so I did not remove them. Instead, I kept every row but added a flag column to mark the most extreme prices. This way I keep 100% of the real data, but a future model in Project 3 can still see which rows are extreme and decide how to handle them.

In [8]:
# Retain all real data, but FLAG the most extreme prices in BOTH close and adj_close.
# quantile(0.999) = the value below which 99.9% of values sit,
# so only the top 0.1% (the extreme spikes like KTOS) get marked.
close_threshold = hsp['close'].quantile(0.999)
adj_threshold   = hsp['adj_close'].quantile(0.999)

# new flag columns: True if the value is above its threshold, else False
hsp['is_extreme_close']     = hsp['close'] > close_threshold
hsp['is_extreme_adj_close'] = hsp['adj_close'] > adj_threshold

# count how many rows got flagged in each
print("close threshold:", round(close_threshold, 2), "| flagged:", hsp['is_extreme_close'].sum())
print("adj_close threshold:", round(adj_threshold, 2), "| flagged:", hsp['is_extreme_adj_close'].sum())

close threshold: 7139.5 | flagged: 20974
adj_close threshold: 8041.55 | flagged: 20974


I decided to retain all the real data but flag the most extreme prices instead of deleting them. Using quantile(0.999), I found the cutoff for the top 0.1% of prices: any close above $7,139.50 and any adjusted close above $8,041.55 is marked as extreme. This flagged 20,974 rows in each column — exactly 0.1% of my ~21 million rows, which is expected since I chose the top 0.1% as the cutoff. I created two new True/False columns, is_extreme_close and is_extreme_adj_close, to label these rows. This way I keep every genuine record but still mark the unusual ones, so a future model can decide how to handle them without me having to delete any real data.

#Part 2 — Create Useful Features


###2.1 — Price Change

In [9]:
# 2.1 price_change = today's close minus the previous day's close, per stock
hsp['price_change'] = hsp.groupby('ticker')['close'].diff()

###Interpretation:

I created price_change, the difference between a stock's closing price today and its close the previous day. It shows, in dollars, how much the price moved in one day. I calculated it per ticker so one company's price never mixes with another's. The first row of each stock is NaN because there's no earlier day to compare to.

### 2.2 daily_return

In [10]:
# 2.2 daily_return = percentage change in close from the previous day, per stock
hsp['daily_return'] = hsp.groupby('ticker')['close'].pct_change()

###Interpretation:
I created daily_return, the percentage change in the closing price from the previous day. Unlike price_change (a dollar amount), this is a percentage, so I can fairly compare cheap and expensive stocks. 0.05 means up 5%, -0.03 means down 3%. First row of each stock is NaN.

###2.2b — Monthly Return

In [11]:
# monthly return = % change in close vs 20 trading days ago (~1 month), per stock
hsp['return_monthly'] = hsp.groupby('ticker')['close'].pct_change(20)

I created return_monthly, the percentage change in the close compared to about one month earlier (20 trading days). While daily_return shows day-to-day movement, this shows the medium-term trend over a month. It's per ticker, and the first ~20 rows of each stock are NaN.

###2.3 — Previous-Day Closing Price

In [12]:
# 2.3 close_lag_1 = the previous trading day's close, per stock
hsp['close_lag_1'] = hsp.groupby('ticker')['close'].shift(1)

I created close_lag_1, which holds yesterday's closing price on each row using shift(1), grouped by ticker. This lets a model see today's and yesterday's price side by side on the same row. First row of each stock is NaN.

### 2.4 — Moving Averages (1 week & 1 month)

In [13]:
# 2.4 moving averages of close, per stock (5 days = 1 week, 20 days = 1 month)
hsp['ma_5']  = hsp.groupby('ticker')['close'].transform(lambda x: x.rolling(5).mean())
hsp['ma_20'] = hsp.groupby('ticker')['close'].transform(lambda x: x.rolling(20).mean())

I created two moving averages of the closing price: ma_5 (average of the last 5 days, about one week) and ma_20 (average of the last 20 days, about one month). A moving average smooths out daily noise and shows the recent trend — the 5-day reacts quickly, the 20-day is slower and steadier. Per ticker; the first few rows of each stock are NaN. (Runs slowly on 21M rows — give it a minute.)

### 2.5 — Volatility (1 month)

In [14]:
# 2.5 volatility_20 = rolling std of daily returns over 20 days (~1 month), per stock
hsp['volatility_20'] = hsp.groupby('ticker')['daily_return'].transform(lambda x: x.rolling(20).std())

I created volatility_20, the standard deviation of daily returns over a 20-day window (about one month). Standard deviation measures how spread out the returns are, so a high value means the stock is bouncing around a lot (risky/unstable) and a low value means it's moving calmly. Per ticker; first ~20 rows of each stock are NaN.

### .6 — Additional Feature 1: Percentage High-Low Range

In [15]:
# 2.6a pct_high_low_range = daily swing (high - low) as a % of close
hsp['pct_high_low_range'] = (hsp['high'] - hsp['low']) / hsp['close']

I created pct_high_low_range, which measures how much a stock's price swung during a single day — the gap between its high and low, divided by its close so it's a percentage. In simple terms, it shows how "wide" the day's trading was: a big value means the price jumped around a lot that day (an unstable, uncertain day), while a small value means it was calm. I divided by the close so it's comparable across cheap and expensive stocks. This is useful for prediction because large intraday swings often come around important events and can signal bigger moves ahead.

### 2.6 a — Additional Feature 1: Volume Change

In [16]:
# volume_change = % change in volume vs the previous day, per stock
hsp['volume_change'] = hsp.groupby('ticker')['volume'].pct_change()

I created volume_change, the percentage change in trading volume compared to the previous day. In simple terms, it shows whether trading activity jumped up or dropped from one day to the next. A sudden spike often means "something is happening" — news, earnings, or big investor interest — which frequently comes right before or during large price moves. It's per ticker, so the first row of each stock is NaN.

### 2.6b Volume Feature 2 — 20-Day Average Volume

In [17]:
# 20-day average volume = sum of last 20 days' volume ÷ 20, per stock (the "normal" level)
hsp['avg_volume_20'] = hsp.groupby('ticker')['volume'].transform(lambda x: x.rolling(20).mean())

I created avg_volume_20, the average trading volume over the last 20 days (about one month). It literally adds up the last 20 days of volume and divides by 20, giving each stock's "normal" activity level. I calculated it per ticker so it reflects each stock's own baseline. The first ~20 rows of each stock are NaN because there aren't 20 earlier days yet.

### 2.6c Volume Feature 3 — Relative Volume

In [18]:
# relative_volume = today's volume ÷ the stock's own 20-day average (1 = normal, 2 = double)
hsp['relative_volume'] = hsp['volume'] / hsp['avg_volume_20']

I created relative_volume, which compares today's volume to the stock's own 20-day average. It answers "is today busier or quieter than normal for this stock?" A value of 1 means a normal day, above 1 means busier than usual, and below 1 means quieter. I built it because raw volume can't be compared between stocks, but this ratio can — every stock is measured against its own baseline.

In [19]:
#check the shaep of data and first few rows
print(hsp.shape)
print(hsp.head(30))


(20973889, 21)
         ticker       open      close  adj_close        low       high  \
13766141      A  32.546494  31.473534  27.494957  28.612303  35.765381   
13766149      A  30.713520  28.880543  25.229753  28.478184  30.758226   
13766150      A  29.551144  31.473534  27.494957  28.657009  31.473534   
13766151      A  30.400572  28.612303  24.995413  28.612303  31.205294   
13766152      A  28.701717  29.372318  25.659359  28.612303  29.998211   
13766153      A  29.238197  29.461731  25.737471  29.148785  29.685265   
13766154      A  29.327610  30.132332  26.323301  29.014664  30.355865   
13766155      A  30.042919  30.177038  26.362358  29.282904  30.713520   
13766156      A  30.177038  30.713520  26.831017  29.953505  31.071173   
13766157      A  31.294706  31.562946  27.573071  30.892345  32.188843   
13766158      A  32.144135  31.831188  27.807402  31.697067  32.680614   
13766159      A  32.367668  32.725323  28.588512  32.322960  33.217094   
13766160      A  32.725

After creating all my new features, I checked the shape and the first 30 rows. The dataset now has 21 columns — the 8 original columns plus the flag columns and all the engineered features (price change, returns, lag, moving averages, volatility, and the volume features). Looking at the first 30 rows, I can see null (NaN) values in the new feature columns at the top. This is expected: features like close_lag_1, ma_5, ma_20, volatility_20, and the returns need earlier days to calculate, so at the very start of a stock's history there isn't enough past data yet. The lag and 1-week features fill in first, while the 20-day (1-month) features stay NaN the longest, until each stock has 20 days of history. I'll handle these expected missing values in the next part.

###1. What does daily_return tell us?
daily_return tells us the percentage change in a stock's closing price from one day to the next — basically, how much it went up or down in a day, as a percent. A value of 0.05 means it rose about 5%, and -0.03 means it fell about 3%. Because it's a percentage, it lets me compare price movement fairly across cheap and expensive stocks.

###2. What does a moving average tell us about a stock?
A moving average tells us the recent trend of a stock's price by smoothing out the daily ups and downs. Instead of reacting to every single day, it averages the last few days (like 5 or 20), so I can see the general direction more clearly. The short one (ma_5, ~1 week) reacts quickly to recent moves, while the longer one (ma_20, ~1 month) shows a slower, steadier trend.

###3. What does high volatility indicate?
High volatility indicates that a stock's price is bouncing around a lot — its daily returns are spread out widely, so the price is unstable and unpredictable in the short term. This usually means higher risk and often happens around big news or uncertain periods. Low volatility means the price is moving calmly and steadily.

###4. Why do lag and rolling features create missing values at the beginning of each stock's history?
Because these features need earlier data that doesn't exist yet at the start. A lag feature (like close_lag_1) looks at the previous day, so the very first day of a stock has no day before it → NaN. A rolling feature (like ma_20 or volatility_20) needs a full window of past days, so until a stock has 20 days of history, there aren't enough values to calculate it → the first ~20 rows are NaN. Since I calculated everything per ticker, this happens fresh at the beginning of each stock's own history, not just once at the top of the dataset.

# Part 3 — Combine and Organize the Data


### 3.1 — Add company info using map (memory-safe):

In [20]:
# add exchange, sector, industry using map instead of merge (memory-light)
lookup = hs.set_index('ticker')          # ticker becomes the index we look up by
hsp['exchange'] = hsp['ticker'].map(lookup['exchange']).astype('category')
hsp['sector']   = hsp['ticker'].map(lookup['sector']).astype('category')
hsp['industry'] = hsp['ticker'].map(lookup['industry']).astype('category')

print("shape:", hsp.shape)

shape: (20973889, 24)


I added the company information (exchange, sector, industry) from the stocks file into the price data. Instead of using merge, I used map, which looks up each row's ticker and fills in the matching value directly. I chose map because merge builds a whole second copy of the 21-million-row table in memory, which crashed the runtime. map doesn't copy the table — it just adds the columns in place — so it uses far less memory. It also can't accidentally duplicate rows the way a merge with repeated keys can, so my row count stays exactly the same (~21M), which confirms the join was clean. I also converted the three new text columns to the category type to save even more memory, since the same sector and exchange names repeat millions of times.

# FROM here project star in jypter

### 3.2 — Handle the feature-created missing values

Decision on feature-created missing values :

The features I created (lags, moving averages, volatility) produce missing values at the start of each stock's history, because they need earlier days that don't exist yet. There were three ways I could handle this:

Option 1 – Fill with 0: replace every missing value with zero. I rejected this because it would be fake data — a 0% return or a $0 moving average never actually happened, and it would mislead the model. The assignment also specifically warns against this.

Option 2 – min_periods: calculate the average using however many days are available so far (day 1 uses 1 value, day 2 averages 2 values, and so on until it reaches the full window). This keeps all the rows and uses real math, but the earliest values are based on only 1–2 days, so they are less stable and not truly a "20-day" average.

Option 3 – Drop the warm-up rows : remove the early rows where the features can't yet be calculated. I chose this because it keeps my data the cleanest — every remaining row has a fully valid, true-window feature, which is exactly what I want for modeling. It only removes the first ~20 rows of each stock, which is less than 1% of the whole dataset, so it's a very small, safe loss in exchange for cleaner, more reliable data.

In [21]:
# these NaNs are each stock's warm-up period
feature_cols = ['price_change','daily_return','return_monthly','close_lag_1',
                'ma_5','ma_20','volatility_20','pct_high_low_range',
                'avg_volume_20','relative_volume','volume_change']

before = len(hsp)
print("rows with any feature NaN:", hsp[feature_cols].isnull().any(axis=1).sum())

# drop the warm-up rows (do NOT fill with zero — that would be fake data)
hsp = hsp.dropna(subset=feature_cols).reset_index(drop=True)
print("dropped:", before - len(hsp), "| remaining:", len(hsp))

rows with any feature NaN: 113555
dropped: 113555 | remaining: 20860334


The missing values in my engineered features aren't errors — they appear because features like close_lag_1, ma_20, and volatility_20 need earlier days that don't exist yet at the very start of each stock's history. I chose to drop these warm-up rows rather than fill them, because filling with zero would create fake returns and averages that never happened, which would mislead a model. Since only the first ~20 rows of each stock are affected, this removes less than 1% of the data — a small, safe loss that leaves every remaining row with fully valid features. I did not replace them with zero, as the assignment warns against.

# 3.3 — Encode Categorical Variables

In [22]:
# fill missing categories as an explicit "UNKNOWN" group so those rows aren't lost,
# then one-hot encode exchange and sector (NOT industry — 136 values = too many columns)
hsp['exchange'] = hsp['exchange'].astype('object').fillna('UNKNOWN')
hsp['sector']   = hsp['sector'].astype('object').fillna('UNKNOWN')

hsp = pd.get_dummies(hsp, columns=['exchange', 'sector'], prefix=['exch', 'sector'])
print("shape after encoding:", hsp.shape)

shape after encoding: (20860334, 37)


Interpretation:

Machine-learning models work with numbers, not text, so I converted my categorical columns (exchange and sector) into numbers using one-hot encoding. First I filled the missing sector values with "UNKNOWN" so those stocks became their own group instead of being lost. Then one-hot encoding created a separate 0/1 column for each category — for example a NYSE stock gets a 1 in the exch_NYSE column and 0 in exch_NASDAQ. I did not encode industry because it has 136 different values, which would add far too many columns. This turns my categories into a model-friendly numeric form without losing their meaning.

Why is one-hot preferable to just numbering (NASDAQ = 1, NYSE = 2)?

One-hot is better because assigning plain numbers accidentally creates a false order and size. If I label NASDAQ = 1 and NYSE = 2, a model would think NYSE is "greater than" or "twice" NASDAQ — but that's meaningless, since exchanges aren't ranked or measured on a scale. This false ordering would mislead the model. One-hot encoding avoids this by giving each category its own separate 0/1 column, so no category is treated as bigger or smaller than another — they're all independent. In short, numbering invents a ranking that doesn't exist, while one-hot keeps every category equal and separate.

# Part 4 — Prepare the Final Dataset

In [23]:
# drop unencoded text column we're not using; keep prices, features, flags, and one-hot columns
stock_prepared = hsp.drop(columns=['industry'])
print("final columns:", list(stock_prepared.columns))
print("shape:", stock_prepared.shape)

final columns: ['ticker', 'open', 'close', 'adj_close', 'low', 'high', 'volume', 'date', 'is_extreme_close', 'is_extreme_adj_close', 'price_change', 'daily_return', 'return_monthly', 'close_lag_1', 'ma_5', 'ma_20', 'volatility_20', 'pct_high_low_range', 'volume_change', 'avg_volume_20', 'relative_volume', 'exch_NASDAQ', 'exch_NYSE', 'sector_BASIC INDUSTRIES', 'sector_CAPITAL GOODS', 'sector_CONSUMER DURABLES', 'sector_CONSUMER NON-DURABLES', 'sector_CONSUMER SERVICES', 'sector_ENERGY', 'sector_FINANCE', 'sector_HEALTH CARE', 'sector_MISCELLANEOUS', 'sector_PUBLIC UTILITIES', 'sector_TECHNOLOGY', 'sector_TRANSPORTATION', 'sector_UNKNOWN']
shape: (20860334, 36)


Interpretation: For my final dataset I kept the identifiers (ticker, date), the core price/volume columns, all my engineered features (returns, lag, moving averages, volatility, high-low range, volume features), the extreme-value flags, and the one-hot encoded exchange/sector columns. I dropped industry because it has 136 text categories that I never encoded, so it can't be used by a model as-is and would only add noise. This gives a clean, fully numeric, model-ready dataset.

# 4.2 — Check the final dataset

In [24]:
print("rows, cols:", stock_prepared.shape)
print("\ndata types:\n", stock_prepared.dtypes)
print("\ntotal missing values:", stock_prepared.isnull().sum().sum())
print("duplicate ticker+date:", stock_prepared.duplicated(subset=['ticker','date']).sum())
print("date range:", stock_prepared['date'].min(), "->", stock_prepared['date'].max())

rows, cols: (20860334, 36)

data types:
 ticker                                     str
open                                   float64
close                                  float64
adj_close                              float64
low                                    float64
high                                   float64
volume                                   int64
date                            datetime64[us]
is_extreme_close                          bool
is_extreme_adj_close                      bool
price_change                           float64
daily_return                           float64
return_monthly                         float64
close_lag_1                            float64
ma_5                                   float64
ma_20                                  float64
volatility_20                          float64
pct_high_low_range                     float64
volume_change                          float64
avg_volume_20                          float64
relative_volume    

Interpretation: Before saving, I checked the final dataset. It has 20,860,334 rows and 36 columns. The data types are correct — numeric features (float/int), boolean 0/1 columns for the flags and one-hot categories, and datetime for date. There are 0 remaining missing values, 0 duplicate ticker+date combinations, and the dates span 1970-01-30 to 2018-08-24. This confirms the data is clean, fully numeric, and ready for modelling in Project 3.

# 4.3 — Save the prepared dataset

In [25]:
# save the final dataset
stock_prepared.to_csv("stock_prepared.csv", index=False)
print("saved stock_prepared.csv")

saved stock_prepared.csv


# Part 5 — Documentation and Findings

1. What data-quality issues did you investigate?

I investigated missing values, duplicate rows, and unusual records. In the company file I found 1,440 stocks with no sector or industry, and one junk row where a header line ("Symbol") was stored as data. In the price file I checked for duplicate ticker+date combinations, negative or zero prices, and negative volume. I also handled the missing values that my own new features created at the start of each stock's history.

2. What decisions did you make about unusual observations?

I decided to retain the unusual high prices (like the very expensive stocks) because I confirmed in Project 1 that they are real prices, not errors. Instead of deleting them, I added flag columns (is_extreme_close and is_extreme_adj_close) to mark the top 0.1% of values, so I keep all the real data but a future model can still see which rows are extreme. I removed the junk "Symbol" record because it was clearly invalid.

3. What new features did you create?

I created several features from the price data: price_change (daily dollar change), daily_return (daily % change), return_monthly (~1-month % change), close_lag_1 (previous day's close), ma_5 and ma_20 (1-week and 1-month moving averages), volatility_20 (1-month volatility), pct_high_low_range (daily price swing as a %), and three volume features (volume_change, avg_volume_20, and relative_volume).

4. Which feature do you think may be useful for future analysis, and why?

I think relative_volume may be very useful. It compares a stock's volume today to its own 20-day average, so it shows when trading is unusually busy for that specific stock. Because it's normalized, it works fairly across cheap and expensive stocks, and sudden jumps in activity often come right before or during big price moves — which is exactly the kind of signal a predictive model can learn from.

5. What does your final prepared dataset contain?

My final dataset has 20,860,334 rows and 36 columns. It contains the identifiers (ticker, date), the core price and volume columns, all my engineered features, the extreme-value flags, and one-hot encoded columns for exchange and sector. It has no missing values and no duplicate ticker+date rows, and the data covers 1970 to 2018. I saved it as stock_prepared.csv for use in Project 3.

6. What additional information or features might be useful when building a predictive model?

A few extra things could help: outside data like company news, earnings reports, or overall market conditions (which explain big price moves the price data alone can't); more technical indicators like RSI or MACD; longer-term features (like a 200-day moving average); and a clear target/label to predict (for example, whether the price goes up or down the next day). Sector or market-wide trends could also give the model useful context.